1- Extract text 

In [2]:
# Install dependencies first (in terminal):
# pip install pymupdf

import fitz  # PyMuPDF
import os

def extract_text_from_pdf(pdf_path):
    """
    Extracts and returns text from a PDF file.

    Parameters:
        pdf_path (str): The file path to the PDF.

    Returns:
        str: All extracted text from the PDF.
    """
    text_content = []

    try:
        # Open the PDF file
        with fitz.open(pdf_path) as pdf_document:
            for page_number in range(len(pdf_document)):
                page = pdf_document[page_number]
                text = page.get_text()
                if text.strip():
                    text_content.append(text)

        return "\n".join(text_content)

    except Exception as e:
        raise RuntimeError(f"Error reading PDF: {e}")


# Example usage
if __name__ == "__main__":
    pdf_file_digital = "krasat.pdf"  # ضع اسم ملف الـ PDF هنا (مثلاً نفس مجلد الكود)
    extracted_text_digital = extract_text_from_pdf(pdf_file_digital)

    # Save to a text file (optional)
    output_txt_digital = os.path.join(os.path.dirname(pdf_file_digital), "output_digital.txt")
    with open(output_txt_digital, "w", encoding="utf-8") as output_file:
        output_file.write(extracted_text_digital)

    print("✅ Text extraction completed for digital PDF. Saved to output_digital.txt")
    print("\n📄 Preview of extracted text:\n")
    print(extracted_text_digital[:1000])  # اطبع أول 1000 حرف فقط


✅ Text extraction completed for digital PDF. Saved to output_digital.txt

📄 Preview of extracted text:

نموذج كراسة الشروط والمواصفات
)(نموذج عام
هـ ، والمعدل بموجب قرار وزير المالية رقم1441/4/12 ) وتاريخ1440( المعتمد بموجب قرار وزير المالية رقم
هـ1445/10/17 ) وتاريخ1156(
2025 ﺗﻮﻓﻴﺮ وﺗﻘﺪﻳﻢ ورش ﻋﻤﻞ ﺗﺪرﻳﺒﻴﺔ ﻣﺼﺎﺣﺒﺔ ﻟﻤﺆﺗﻤﺮ إدارة اﻟﻤﺨﺎﻃﺮ واﻟﻄﻮارئ واﺳﺘﻤﺮارﻳﺔ اﻷﻋﻤﺎل :اسم المنافسة
33-25 :رقم الكراسة
22/01/1447 :تاريخ طرح الكراسة
المملكة العربية السعودية
*إسم اإلدارة : جامعة الملك عبدالعزيز مركز المبدعون للدراسات
إسم النموذج : ﺗﻮﻓﻴﺮ وﺗﻘﺪﻳﻢ ورش ﻋﻤﻞ ﺗﺪرﻳﺒﻴﺔ ﻣﺼﺎﺣﺒﺔ ﻟﻤﺆﺗﻤﺮ إدارة اﻟﻤﺨﺎﻃﺮ واﻟﻄﻮارئ
2025 واﺳﺘﻤﺮارﻳﺔ اﻷﻋﻤﺎل
رقم النسخة   : األولى
:  تاريخ اإلصدار
33-25 :  رقم الكراسة
 ص9:48 2025/‏7/‏21كراسة الشروط والمواصفات
https://tenders.etimad.sa/Tender/PrintConditionsTemplateRfp?STenderId=vT686*%40%40**iAo3PFZToy8QRuVg%3D%3D
1/31

الفهرس
مقدمة : القسم األول
تعريفات1
تعريف عن المنافسة2
تكاليف وثائق المنافسة3
المواعيد المتعلقة بالمنافسة4
أهلية مقدمي العروض5
السجالت والتراخيص النظامية6
ممثل الجهة الحكوم

2- Extract text using OCR


In [2]:
# -*- coding: utf-8 -*-
import re, json, unicodedata
from pathlib import Path

IN_TXT   = Path("rfp_dataset.txt")            # ملفك الخام
OUT_JSON = Path("rfp_dataset.cleaned.jsonl")  # ناتج للتدريب
MAX_CHARS = 3500                               # حجم كل مثال (قرّبيه حسب الحاجة)

# ---------- تنظيف عربي ----------
DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06ED]")
CTRL = re.compile(r"[\u200e\u200f\u202a-\u202e\ufeff]")
NBSP = "\u00A0"

HEADER_FOOTER_PATTERNS = [
    r"^\s*وزارة\s+المالية\s*$",
    r"^\s*المركز\s+الوطني\s+للأرصاد\s*$",
    r"^\s*صفحة\s+\d+\s*$",
    r"^\s*Page\s+\d+\s*$",
]

def normalize_ar(s: str) -> str:
    s = unicodedata.normalize("NFKC", s)
    s = CTRL.sub("", s)
    s = s.replace("ـ","")
    s = DIAC.sub("", s)
    # تطبيع الحروف
    s = (s.replace("أ","ا").replace("إ","ا").replace("آ","ا")
           .replace("ى","ي").replace("ة","ه").replace("ؤ","و").replace("ئ","ي"))
    # NBSP -> space
    s = s.replace(NBSP, " ")
    # توحيد أرقام عربية/هندية (اختياري)
    trans_digits = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
    s = s.translate(trans_digits)
    # مسافات وعلامات ترقيم
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\s*([،,:;.\-–—()])\s*", r" \1 ", s)
    s = re.sub(r"\s{2,}", " ", s).strip()
    return s

def drop_headers_footers(raw: str) -> str:
    kept = []
    for ln in raw.splitlines():
        # شطب رؤوس/ذيول شائعة
        if any(re.search(p, ln) for p in HEADER_FOOTER_PATTERNS):
            continue
        # شطب أسطر ترقيم الصفحات مثل "-59" أو "- 60"
        if re.match(r"^\s*-\s*\d+\s*$", ln):
            continue
        kept.append(ln)
    # دمج زائد
    txt = "\n".join(kept)
    # إزالة فواصل مطولة
    txt = re.sub(r"-{8,}", "\n", txt)
    # ضغط الأسطر الفارغة
    txt = re.sub(r"\n{3,}", "\n\n", txt)
    return txt

def clean_text(raw: str) -> str:
    # أبقي النسخة “raw” للأرشفة لو تحبي، وهنا ننظّف للمطابقة/التدريب
    return normalize_ar(drop_headers_footers(raw))

# ---------- تقطيع نص طويل ----------
def chunk_text(s: str, max_chars: int):
    s = s.strip()
    if len(s) <= max_chars:
        return [s] if s else []
    chunks, cur, tot = [], [], 0
    # نحاول نحافظ على الفقرات قدر الإمكان
    for seg in re.split(r"(\n{2,})", s):
        if tot + len(seg) > max_chars and cur:
            chunks.append("".join(cur).strip())
            cur, tot = [seg], len(seg)
        else:
            cur.append(seg); tot += len(seg)
    if cur:
        chunks.append("".join(cur).strip())
    return [c for c in chunks if c]

# ---------- تقسيم الملف إلى منافسات ----------
# نقسم عند ظهور "اسم المنافسة:" كسطر/بداية بلوك
COMP_SPLIT = re.compile(r"(?=^اسم\s+المنافس[هة]\s*:\s*)", re.MULTILINE)

# بداية/نهاية الأقسام (مرنة)
SEC7_START = re.compile(r"القسم\s*السابع.*?نطاق\s*العمل", re.IGNORECASE | re.DOTALL)
SEC10_START = re.compile(r"القسم\s*العاشر.*?الشروط\s*الخاصه|القسم\s*العاشر\s*:\s*الشروط\s*الخاصه", re.IGNORECASE | re.DOTALL)
NEXT_SECTION = re.compile(r"\n\s*القسم\s+(?:الثامن|التاسع|العاشر|الحادي\s*عشر|الحاديه\s*عشر)\b", re.IGNORECASE)

NAME_RE = re.compile(r"^اسم\s+المنافس[هة]\s*:\s*(.+)", re.IGNORECASE | re.MULTILINE)

def find_name(block: str) -> str:
    m = NAME_RE.search(block)
    if m:
        return m.group(1).strip()
    # fallback: أول سطر غير فارغ
    first = next((ln.strip() for ln in block.splitlines() if ln.strip()), "")
    return first[:150] or "بدون_اسم"

def slice_between(block: str, start_re, end_re):
    ms = start_re.search(block)
    if not ms:
        return ""
    start = ms.end()
    me = end_re.search(block, start)
    end = me.start() if me else len(block)
    return block[start:end].strip()

def extract_fields(block_raw: str):
    # نشتغل على نسخة “تنظيف خفيف” للمطابقة لكن نُرجِع النص كما هو تقريبًا
    block_for_match = clean_text(block_raw)
    name = find_name(block_raw)

    scope = slice_between(block_for_match, SEC7_START, NEXT_SECTION)
    spec  = slice_between(block_for_match, SEC10_START, NEXT_SECTION)

    # fallback بعناوين أبسط لو أحدهما فاضي
    if not scope:
        m = re.search(r"نطاق\s*عمل\s*المشروع.*?(?=\n\s*القسم\s+|$)", block_for_match, re.IGNORECASE | re.DOTALL)
        scope = (m.group(0).strip() if m else "")
    if not spec:
        m = re.search(r"الشروط\s*الخاصه.*?(?=\n\s*القسم\s+|$)", block_for_match, re.IGNORECASE | re.DOTALL)
        spec = (m.group(0).strip() if m else "")

    # تنظيف نهائي للاستخدام التدريبي
    scope_clean = clean_text(scope)
    spec_clean  = clean_text(spec)
    return name, scope_clean, spec_clean

# ---------- بناء JSONL ----------
def to_examples(competition: str, scope: str, spec: str):
    ex = []
    for ch in chunk_text(scope, MAX_CHARS):
        ex.append({"prompt": f"ما هو نطاق العمل لمنافسة: {competition}؟", "completion": ch})
    for ch in chunk_text(spec, MAX_CHARS):
        ex.append({"prompt": f"ما هي الشروط الخاصة لمنافسة: {competition}؟", "completion": ch})
    return ex

def main():
    raw = IN_TXT.read_text(encoding="utf-8", errors="ignore")
    # نفصل إلى منافسات
    blocks = [b.strip() for b in COMP_SPLIT.split(raw) if b.strip()]
    # نحافظ فقط على البلوكات اللي تبدأ بـ اسم المنافسة
    blocks = [b for b in blocks if b.startswith("اسم المنافسة") or b.startswith("اسم المنافسه")]

    total = 0
    with OUT_JSON.open("w", encoding="utf-8") as out:
        for b in blocks:
            name, scope, spec = extract_fields(b)
            # لو ما لقي أي قسم، نتخطّى
            if not scope and not spec:
                continue
            for rec in to_examples(name, scope, spec):
                out.write(json.dumps(rec, ensure_ascii=False) + "\n")
                total += 1
    print(f"✅ Done. Wrote {total} examples → {OUT_JSON}")

if __name__ == "__main__":
    main()


✅ Done. Wrote 6 examples → rfp_dataset.cleaned.jsonl


_____________________________________________________________________________________________________

_____________________________________________________________________________________________________